# 06. Aleppo Building Damage: Temporal Analysis & Damage Causes

This notebook analyzes the temporal progression of building damage in Aleppo during the Syrian conflict, examining damage patterns over time and categorizing damage by severity and type.

**Data Source:** UNOSAT Damage Assessment Database
- Satellite-based damage detection
- Multiple assessment dates
- Neighborhood-level analysis

## Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import leafmap
import geopandas as gpd
from datetime import datetime
import warnings

warnings.filterwarnings("ignore")

# Set plotting styles
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

## Data Loading & Preparation

Load UNOSAT damage assessment data for Aleppo. This example uses a sample dataset structure; adapt the path to your actual data source.

In [2]:
# Example: Loading from a CSV file with UNOSAT damage assessment data
# You may need to adjust the file path based on your actual data location

# For demonstration, we create a sample dataset structure
# In practice, this would load from your UNOSAT or similar damage assessment database

# Example data structure:
# - neighborhood: Neighborhood name
# - assessment_date: Date of assessment
# - buildings_damaged: Count of damaged buildings
# - buildings_total: Total buildings in neighborhood
# - damage_severity: 'destroyed', 'severely_damaged', 'moderately_damaged', 'lightly_damaged'
# - latitude, longitude: Spatial coordinates

# Load your actual data here
# df_damage = pd.read_csv('../Data/aleppo_damage_assessment.csv')
# gdf_damage = gpd.read_file('../Data/aleppo_neighborhoods.geojson')

print("Data loading section: Replace with your actual UNOSAT damage assessment data")

Data loading section: Replace with your actual UNOSAT damage assessment data


## Damage Severity Classification

Categorize damage based on percentage of buildings affected and severity levels.

In [3]:
def categorize_damage_severity(damage_percentage):
    """
    Categorize damage severity based on percentage of buildings damaged.

    Parameters:
    -----------
    damage_percentage : float
        Percentage of buildings damaged (0-100)

    Returns:
    --------
    str : Severity category
    """
    if damage_percentage < 10:
        return "Low (0-10%)"
    elif damage_percentage < 25:
        return "Moderate (10-25%)"
    elif damage_percentage < 50:
        return "High (25-50%)"
    else:
        return "Severe (50%+)"


def categorize_damage_type(damage_code):
    """
    Categorize damage type based on assessment codes.

    UNOSAT Damage Categories:
    - D1: Destroyed
    - D2: Severely Damaged
    - D3: Moderately Damaged
    - D4: Lightly Damaged
    - U: Undamaged
    """
    damage_map = {
        "D1": "Destroyed",
        "D2": "Severely Damaged",
        "D3": "Moderately Damaged",
        "D4": "Lightly Damaged",
        "U": "Undamaged",
    }
    return damage_map.get(damage_code, "Unknown")


print("Damage classification functions defined")

Damage classification functions defined


## Temporal Analysis Functions

Functions to analyze damage progression over time.

In [4]:
def calculate_damage_progression(
    df, date_column="assessment_date", damage_column="buildings_damaged"
):
    """
    Calculate cumulative and new damage over time.

    Parameters:
    -----------
    df : pd.DataFrame
        Damage assessment data with dates
    date_column : str
        Column name for assessment dates
    damage_column : str
        Column name for damage counts

    Returns:
    --------
    pd.DataFrame : Time series of damage progression
    """
    # Group by date and sum damage
    damage_by_date = df.groupby(date_column)[damage_column].sum().sort_index()

    # Calculate cumulative damage
    cumulative_damage = damage_by_date.cumsum()

    # Calculate new damage (differences between dates)
    new_damage = damage_by_date.diff().fillna(damage_by_date.iloc[0])

    result = pd.DataFrame(
        {
            "date": damage_by_date.index,
            "new_damage": new_damage.values,
            "cumulative_damage": cumulative_damage.values,
        }
    )

    return result


def identify_damage_hotspots(
    df, neighborhood_column="neighborhood", damage_column="buildings_damaged", top_n=15
):
    """
    Identify most-damaged neighborhoods.

    Parameters:
    -----------
    df : pd.DataFrame
        Damage assessment data
    neighborhood_column : str
        Column name for neighborhood names
    damage_column : str
        Column name for damage counts
    top_n : int
        Number of top neighborhoods to return

    Returns:
    --------
    pd.DataFrame : Top damaged neighborhoods
    """
    # Group by neighborhood and sum damage
    hotspots = (
        df.groupby(neighborhood_column)[damage_column]
        .sum()
        .sort_values(ascending=False)
        .head(top_n)
    )

    return pd.DataFrame(
        {"neighborhood": hotspots.index, "buildings_damaged": hotspots.values}
    )


print("Temporal analysis functions defined")

Temporal analysis functions defined


## Visualization Functions

Create comprehensive visualizations of damage patterns.

In [5]:
def plot_damage_over_time(damage_progression):
    """
    Plot cumulative and new damage over time.
    """
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

    # Cumulative damage
    ax1.plot(
        damage_progression["date"],
        damage_progression["cumulative_damage"],
        marker="o",
        linewidth=2,
        color="darkred",
    )
    ax1.fill_between(
        damage_progression["date"],
        damage_progression["cumulative_damage"],
        alpha=0.3,
        color="red",
    )
    ax1.set_title(
        "Cumulative Building Damage Over Time", fontsize=14, fontweight="bold"
    )
    ax1.set_ylabel("Cumulative Buildings Damaged", fontsize=12)
    ax1.grid(True, alpha=0.3)

    # New damage per assessment
    ax2.bar(
        damage_progression["date"],
        damage_progression["new_damage"],
        color="steelblue",
        alpha=0.7,
    )
    ax2.set_title(
        "New Damage Detections per UNOSAT Assessment Date",
        fontsize=14,
        fontweight="bold",
    )
    ax2.set_xlabel("Assessment Date", fontsize=12)
    ax2.set_ylabel("Buildings Newly Damaged", fontsize=12)
    ax2.grid(True, alpha=0.3, axis="y")

    plt.tight_layout()
    return fig


def plot_top_damaged_neighborhoods(hotspots, top_n=15):
    """
    Plot bar chart of most-damaged neighborhoods.
    """
    fig, ax = plt.subplots(figsize=(10, 8))

    hotspots_sorted = hotspots.sort_values("buildings_damaged")

    ax.barh(
        hotspots_sorted["neighborhood"],
        hotspots_sorted["buildings_damaged"],
        color="#A64D33",
        alpha=0.8,
    )
    ax.set_xlabel("Buildings Damaged", fontsize=12)
    ax.set_title(
        f"{top_n} Most-Damaged Neighborhoods in Aleppo", fontsize=14, fontweight="bold"
    )
    ax.grid(True, alpha=0.3, axis="x")

    plt.tight_layout()
    return fig


def plot_damage_severity_distribution(df, damage_pct_column="damage_percentage"):
    """
    Plot histogram of damage severity distribution across neighborhoods.
    """
    fig, ax = plt.subplots(figsize=(10, 6))

    damage_pcts = df[damage_pct_column]
    mean_damage = damage_pcts.mean()
    median_damage = damage_pcts.median()

    ax.hist(damage_pcts, bins=50, color="#A64D33", alpha=0.7, edgecolor="black")
    ax.axvline(
        mean_damage,
        color="darkred",
        linestyle="--",
        linewidth=2,
        label=f"Mean = {mean_damage:.1f}%",
    )
    ax.axvline(
        median_damage,
        color="darkblue",
        linestyle=":",
        linewidth=2,
        label=f"Median = {median_damage:.1f}%",
    )

    ax.set_xlabel("% of Buildings Damaged", fontsize=12)
    ax.set_ylabel("Number of Neighborhoods", fontsize=12)
    ax.set_title(
        "Distribution of Neighborhood-Level Building Damage, Aleppo",
        fontsize=14,
        fontweight="bold",
    )
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3, axis="y")

    plt.tight_layout()
    return fig


def plot_damage_by_severity_type(df, severity_column="damage_severity"):
    """
    Plot distribution of damage by type/severity category.
    """
    fig, ax = plt.subplots(figsize=(10, 6))

    severity_counts = df[severity_column].value_counts().sort_values(ascending=True)

    colors = ["#90EE90", "#FFD700", "#FFA500", "#DC143C"]
    ax.barh(
        severity_counts.index,
        severity_counts.values,
        color=colors[: len(severity_counts)],
    )
    ax.set_xlabel("Number of Neighborhoods", fontsize=12)
    ax.set_title(
        "Damage Classification by Severity Level", fontsize=14, fontweight="bold"
    )
    ax.grid(True, alpha=0.3, axis="x")

    plt.tight_layout()
    return fig


print("Visualization functions defined")

Visualization functions defined


## Interactive Map Creation

Create interactive Leaflet maps for spatial analysis.

In [6]:
def create_damage_map(
    gdf, damage_column="damage_percentage", title="Building Damage by Neighborhood"
):
    """
    Create an interactive map of damage by neighborhood.

    Parameters:
    -----------
    gdf : geopandas.GeoDataFrame
        GeoDataFrame with geometries and damage data
    damage_column : str
        Column name for damage percentage
    title : str
        Map title
    """
    m = leafmap.Map(center=(36.20, 37.15), zoom=11)

    # Add choropleth layer
    m.add_geopandas_layer(
        gdf,
        name="Damage by Neighborhood",
        show=True,
        column=damage_column,
        cmap="YlOrRd",
        legend_title="% Damaged",
    )

    m.add_title(title, font_size=16)
    return m


def create_damage_timeline_map(gdf_list, dates_list, damage_column="damage_percentage"):
    """
    Create a map with layers for different assessment dates.

    Parameters:
    -----------
    gdf_list : list of geopandas.GeoDataFrame
        List of GeoDataFrames for different dates
    dates_list : list of str
        List of dates corresponding to each GeoDataFrame
    damage_column : str
        Column name for damage percentage
    """
    m = leafmap.Map(center=(36.20, 37.15), zoom=11)

    for gdf, date in zip(gdf_list, dates_list):
        m.add_geopandas_layer(
            gdf,
            name=f"Damage - {date}",
            show=False,
            column=damage_column,
            cmap="YlOrRd",
            legend_title="% Damaged",
        )

    return m


print("Map creation functions defined")

Map creation functions defined


## Mapbox Setup & Configuration

Configure Mapbox for enhanced interactive visualizations.

In [7]:
# Set up Mapbox token for enhanced map visualizations
import os

MAPBOX_TOKEN = "pk.eyJ1Ijoia2hhbGVkYWxhbmplcnkiLCJhIjoiY21zOXJtdGE3MHM5NDJ3b2RhZDhlN3RzdiJ9.u-954HDL1cS8ZVODaKr3IQ"

# Configure leafmap to use Mapbox by setting environment variable
os.environ["MAPBOX_ACCESS_TOKEN"] = MAPBOX_TOKEN

# Verify token is set
if os.environ.get("MAPBOX_ACCESS_TOKEN"):
    print("✓ Mapbox token configured successfully")
    print("Ready to create interactive visualizations with Mapbox basemaps")
else:
    print("⚠ Warning: Mapbox token not set")

✓ Mapbox token configured successfully
Ready to create interactive visualizations with Mapbox basemaps


In [8]:
def create_damage_map_mapbox(
    gdf=None,
    damage_column="damage_percentage",
    title="Aleppo Building Damage Assessment",
    basemap="Esri.WorldImagery",
):
    """
    Create an interactive damage map with satellite/street basemap.

    Parameters:
    -----------
    gdf : geopandas.GeoDataFrame, optional
        GeoDataFrame with geometries and damage data
    damage_column : str
        Column name for damage percentage
    title : str
        Map title
    basemap : str
        Basemap style - options:
        - 'Esri.WorldImagery' (satellite, default)
        - 'Esri.WorldStreetMap' (street map)
        - 'CartoDB.Positron' (light map)
        - 'CartoDB.DarkMatter' (dark map)
        - 'OpenStreetMap.Mapnik' (OSM)

    Returns:
    --------
    leafmap.Map : Interactive map object
    """
    # Create map centered on Aleppo with selected basemap
    m = leafmap.Map(center=(36.20, 37.15), zoom=11, basemap=basemap)

    # Add damage layers if geodata is provided
    if gdf is not None and damage_column in gdf.columns:
        m.add_geopandas_layer(
            gdf,
            name="Damage by Neighborhood",
            show=True,
            column=damage_column,
            cmap="YlOrRd",
            legend_title="% Buildings Damaged",
            opacity=0.7,
        )

    # Add title
    m.add_title(title, font_size=16)

    return m


def create_damage_analysis_dashboard(df, gdf=None):
    """
    Create a comprehensive damage analysis dashboard with map and visualizations.

    Parameters:
    -----------
    df : pd.DataFrame
        Damage assessment data
    gdf : geopandas.GeoDataFrame, optional
        GeoDataFrame with geometries for choropleth

    Returns:
    --------
    dict : Contains map and summary statistics
    """
    # Create interactive map
    if gdf is not None:
        m = create_damage_map_mapbox(gdf, damage_column="damage_percentage")
    else:
        m = create_damage_map_mapbox()

    # Calculate statistics
    stats = {
        "total_neighborhoods": len(df["neighborhood"].unique()),
        "total_damage": df["buildings_damaged"].sum(),
        "avg_damage_pct": (
            df["damage_percentage"].mean() if "damage_percentage" in df.columns else 0
        ),
        "max_damage_neighborhood": (
            df.loc[df["buildings_damaged"].idxmax(), "neighborhood"]
            if "buildings_damaged" in df.columns
            else "N/A"
        ),
    }

    return {"map": m, "statistics": stats}


print("✓ Mapbox-enhanced map functions updated with valid leafmap basemaps")

✓ Mapbox-enhanced map functions updated with valid leafmap basemaps


## Analysis Workflow Example

Complete workflow for analyzing damage progression and causes (when data is loaded).

In [9]:
# STEP 1: Load your UNOSAT damage assessment data
# Uncomment and adapt the following lines to your data source:

# df_damage = pd.read_csv('../Data/aleppo_damage_assessment.csv', parse_dates=['assessment_date'])
# gdf_damage = gpd.read_file('../Data/aleppo_neighborhoods.geojson')

print("Step 1: Data loading section ready")
print(
    "Expected columns: neighborhood, assessment_date, buildings_damaged, buildings_total, damage_severity"
)

Step 1: Data loading section ready
Expected columns: neighborhood, assessment_date, buildings_damaged, buildings_total, damage_severity


In [10]:
# STEP 2: Data preparation and feature engineering
# Uncomment after loading data:

# # Calculate damage percentage
# df_damage['damage_percentage'] = (df_damage['buildings_damaged'] / df_damage['buildings_total']) * 100

# # Categorize by severity
# df_damage['severity_category'] = df_damage['damage_percentage'].apply(categorize_damage_severity)

# # Categorize by damage type
# df_damage['damage_type'] = df_damage['damage_code'].apply(categorize_damage_type)

print("Step 2: Feature engineering template ready")
print("Will add: damage_percentage, severity_category, damage_type columns")

Step 2: Feature engineering template ready
Will add: damage_percentage, severity_category, damage_type columns


In [11]:
# STEP 3: Temporal analysis
# Uncomment after data preparation:

# damage_progression = calculate_damage_progression(df_damage)
# print("\nDamage Progression Over Time:")
# print(damage_progression)

# # Plot temporal trends
# plot_damage_over_time(damage_progression)
# plt.show()

print("Step 3: Temporal analysis template ready")

Step 3: Temporal analysis template ready


In [12]:
# STEP 4: Identify hotspots and problem areas
# Uncomment after data preparation:

# hotspots = identify_damage_hotspots(df_damage)
# print("\nTop 15 Most-Damaged Neighborhoods:")
# print(hotspots)

# # Visualize hotspots
# plot_top_damaged_neighborhoods(hotspots, top_n=15)
# plt.show()

print("Step 4: Hotspot identification template ready")

Step 4: Hotspot identification template ready


In [13]:
# STEP 5: Analyze damage severity distribution
# Uncomment after data preparation:

# plot_damage_severity_distribution(df_damage)
# plt.show()

# # Analyze by damage type
# damage_by_type = df_damage['damage_type'].value_counts()
# print("\nDamage Distribution by Type:")
# print(damage_by_type)

# plot_damage_by_severity_type(df_damage)
# plt.show()

print("Step 5: Severity distribution template ready")

Step 5: Severity distribution template ready


In [14]:
# STEP 6: Create spatial visualizations
# Uncomment after loading geodata:

# # Merge damage data with geometries
# gdf_damage_merged = gdf_damage.merge(df_damage.groupby('neighborhood').agg({
#     'damage_percentage': 'mean',
#     'buildings_damaged': 'sum',
#     'damage_type': lambda x: x.mode()[0] if len(x.mode()) > 0 else 'Unknown'
# }).reset_index(), left_on='name', right_on='neighborhood', how='left')

# # Create interactive map
# m = create_damage_map(gdf_damage_merged, damage_column='damage_percentage')
# m.show()

print("Step 6: Spatial visualization template ready")

Step 6: Spatial visualization template ready


## Interactive Map Visualization with Mapbox

Display interactive damage assessment map with visualizations.

In [15]:
# Create interactive map visualization window
# This cell displays the damage assessment map with satellite imagery basemap

# Display map with Esri satellite basemap
m_satellite = create_damage_map_mapbox(
    gdf=None,  # Will use after data is loaded
    damage_column="damage_percentage",
    title="Aleppo Building Damage Assessment - Satellite View",
    basemap="Esri.WorldImagery",
)

print("✓ Interactive Satellite Map Created")
print("📍 Centered on Aleppo, Syria")
print("🔍 Zoom in/out to explore neighborhoods")
print("🎨 Layers panel (top right) to toggle visualizations")
m_satellite

The ipyleaflet map does not support titles.
✓ Interactive Satellite Map Created
📍 Centered on Aleppo, Syria
🔍 Zoom in/out to explore neighborhoods
🎨 Layers panel (top right) to toggle visualizations


Map(center=[36.2, 37.15], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

In [16]:
# Alternative: Street view map
# Uncomment to display street-level map instead

# m_streets = create_damage_map_mapbox(
#     gdf=None,
#     damage_column='damage_percentage',
#     title='Aleppo Building Damage Assessment - Street View',
#     basemap='Esri.WorldStreetMap'
# )
#
# print("✓ Street Map Created")
# m_streets

# Other basemap options to try:
# 'CartoDB.Positron' - Light map
# 'CartoDB.DarkMatter' - Dark map
# 'OpenStreetMap.Mapnik' - OpenStreetMap

print("✓ Alternative maps available (uncomment to display)")

✓ Alternative maps available (uncomment to display)


## Complete Dashboard: Map + Analysis

Combined interactive window showing map with synchronized analysis visualizations.

In [17]:
def create_integrated_dashboard(df, gdf=None, figsize=(16, 10)):
    """
    Create an integrated dashboard with map and multiple visualizations.

    Parameters:
    -----------
    df : pd.DataFrame
        Damage assessment data
    gdf : geopandas.GeoDataFrame, optional
        GeoDataFrame with geometries
    figsize : tuple
        Figure size for subplots

    Returns:
    --------
    Displays map and visualizations side-by-side
    """
    from IPython.display import display, HTML

    # Create figure with subplots for visualizations
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=figsize)

    # 1. Damage Distribution Histogram
    if "damage_percentage" in df.columns:
        damage_pcts = df["damage_percentage"].dropna()
        ax1.hist(damage_pcts, bins=30, color="#A64D33", alpha=0.7, edgecolor="black")
        ax1.axvline(
            damage_pcts.mean(),
            color="darkred",
            linestyle="--",
            linewidth=2,
            label=f"Mean: {damage_pcts.mean():.1f}%",
        )
        ax1.set_title(
            "Damage Distribution Across Neighborhoods", fontsize=12, fontweight="bold"
        )
        ax1.set_xlabel("% Buildings Damaged")
        ax1.set_ylabel("Number of Neighborhoods")
        ax1.legend()
        ax1.grid(True, alpha=0.3)

    # 2. Top 10 Hotspots
    if "buildings_damaged" in df.columns and "neighborhood" in df.columns:
        top_10 = (
            df.groupby("neighborhood")["buildings_damaged"]
            .sum()
            .nlargest(10)
            .sort_values()
        )
        ax2.barh(range(len(top_10)), top_10.values, color="#A64D33", alpha=0.7)
        ax2.set_yticks(range(len(top_10)))
        ax2.set_yticklabels(top_10.index, fontsize=9)
        ax2.set_title(
            "Top 10 Most-Damaged Neighborhoods", fontsize=12, fontweight="bold"
        )
        ax2.set_xlabel("Buildings Damaged")
        ax2.grid(True, alpha=0.3, axis="x")

    # 3. Damage by Type (if available)
    if "damage_type" in df.columns:
        damage_type_counts = df["damage_type"].value_counts()
        colors_map = {
            "Destroyed": "#DC143C",
            "Severely Damaged": "#FFA500",
            "Moderately Damaged": "#FFD700",
            "Lightly Damaged": "#90EE90",
        }
        colors = [colors_map.get(dt, "#808080") for dt in damage_type_counts.index]
        ax3.bar(
            range(len(damage_type_counts)),
            damage_type_counts.values,
            color=colors,
            alpha=0.7,
        )
        ax3.set_xticks(range(len(damage_type_counts)))
        ax3.set_xticklabels(
            damage_type_counts.index, rotation=45, ha="right", fontsize=9
        )
        ax3.set_title("Damage Classification by Type", fontsize=12, fontweight="bold")
        ax3.set_ylabel("Count")
        ax3.grid(True, alpha=0.3, axis="y")

    # 4. Temporal Progression (if available)
    if "assessment_date" in df.columns and "buildings_damaged" in df.columns:
        damage_by_date = (
            df.groupby("assessment_date")["buildings_damaged"].sum().sort_index()
        )
        ax4.plot(
            damage_by_date.index,
            damage_by_date.cumsum(),
            marker="o",
            color="darkred",
            linewidth=2,
        )
        ax4.fill_between(
            damage_by_date.index, damage_by_date.cumsum(), alpha=0.3, color="red"
        )
        ax4.set_title("Cumulative Damage Over Time", fontsize=12, fontweight="bold")
        ax4.set_xlabel("Assessment Date")
        ax4.set_ylabel("Cumulative Buildings Damaged")
        ax4.grid(True, alpha=0.3)
        plt.setp(ax4.xaxis.get_majorticklabels(), rotation=45, ha="right")

    plt.tight_layout()
    plt.show()

    # Display statistics panel
    stats_html = f"""
    <div style='background-color: #f0f0f0; padding: 15px; border-radius: 5px; margin: 10px 0;'>
        <h3 style='margin-top: 0; color: #333;'>📊 Damage Assessment Summary</h3>
        <table style='width: 100%; font-family: Arial; font-size: 11pt;'>
            <tr style='background-color: #ddd;'>
                <td style='padding: 8px; font-weight: bold;'>Total Neighborhoods</td>
                <td style='padding: 8px;'>{len(df['neighborhood'].unique()) if 'neighborhood' in df.columns else 'N/A'}</td>
            </tr>
            <tr>
                <td style='padding: 8px; font-weight: bold;'>Total Buildings Damaged</td>
                <td style='padding: 8px;'>{int(df['buildings_damaged'].sum()) if 'buildings_damaged' in df.columns else 'N/A':,}</td>
            </tr>
            <tr style='background-color: #ddd;'>
                <td style='padding: 8px; font-weight: bold;'>Average Damage %</td>
                <td style='padding: 8px;'>{df['damage_percentage'].mean():.1f}% if 'damage_percentage' in df.columns else 'N/A'</td>
            </tr>
            <tr>
                <td style='padding: 8px; font-weight: bold;'>Assessment Dates</td>
                <td style='padding: 8px;'>{df['assessment_date'].nunique() if 'assessment_date' in df.columns else 'N/A'}</td>
            </tr>
        </table>
    </div>
    """

    display(HTML(stats_html))


# Create dashboard with sample/loaded data
# Uncomment after loading df_damage:
#
# display(HTML("<h2>🗺️ Interactive Damage Assessment Dashboard</h2>"))
#
# # Show map first
# print("Map View:")
# m = create_damage_map_mapbox(title='Aleppo - Building Damage Assessment')
# m.show()
#
# # Show analysis visualizations
# print("\nDetailed Analysis:")
# create_integrated_dashboard(df_damage, gdf=None)

print("Integrated dashboard function defined and ready to use")

Integrated dashboard function defined and ready to use


## Quick Start: Using the Interactive Map Visualizations

### Map Controls & Features:
- **Zoom**: Scroll wheel or +/- buttons (top left)
- **Pan**: Click and drag to move around
- **Basemap Toggle**: Select different map styles (satellite, streets, dark, light)
- **Layers Panel**: Top right corner to toggle data layers on/off
- **Fullscreen**: Expand map to fill screen
- **Export**: Save map view as image

### Available Basemap Options:
```python
# Satellite imagery (high-res)
m = create_damage_map_mapbox(basemap='Esri.WorldImagery')

# Street map (detailed)
m = create_damage_map_mapbox(basemap='Esri.WorldStreetMap')

# Light map (minimal)
m = create_damage_map_mapbox(basemap='CartoDB.Positron')

# Dark map (night style)
m = create_damage_map_mapbox(basemap='CartoDB.DarkMatter')

# OpenStreetMap
m = create_damage_map_mapbox(basemap='OpenStreetMap.Mapnik')

# USGS Topo
m = create_damage_map_mapbox(basemap='USGS.USTopo')
```

### Complete Workflow:
```python
# 1. Load your UNOSAT damage data
df_damage = pd.read_csv('path/to/aleppo_damage_data.csv', parse_dates=['assessment_date'])
gdf_damage = gpd.read_file('path/to/aleppo_neighborhoods.geojson')

# 2. Prepare damage metrics
df_damage['damage_percentage'] = (df_damage['buildings_damaged'] / df_damage['buildings_total']) * 100
df_damage['severity'] = df_damage['damage_percentage'].apply(categorize_damage_severity)

# 3. Create interactive map
m = create_damage_map_mapbox(
    gdf=gdf_damage, 
    damage_column='damage_percentage',
    basemap='Esri.WorldImagery',
    title='Aleppo - Building Damage Assessment'
)
m.show()

# 4. Generate full dashboard with analysis
create_integrated_dashboard(df_damage, gdf=gdf_damage)
```

### Tips:
- Use **Esri.WorldImagery** for damage pattern context with satellite view
- Use **Esri.WorldStreetMap** for neighborhood boundary reference
- Adjust map **opacity** in layer properties to see underlying basemap

## Statistical Summary

Generate statistical summaries of damage patterns.

In [18]:
def generate_damage_report(df):
    """
    Generate a comprehensive damage report with statistics.

    Parameters:
    -----------
    df : pd.DataFrame
        Damage assessment data with damage_percentage column
    """
    report = {
        "Total Neighborhoods": len(df["neighborhood"].unique()),
        "Total Buildings Damaged": df["buildings_damaged"].sum(),
        "Mean Damage %": df["damage_percentage"].mean(),
        "Median Damage %": df["damage_percentage"].median(),
        "Std Dev Damage %": df["damage_percentage"].std(),
        "Max Damage %": df["damage_percentage"].max(),
        "Min Damage %": df["damage_percentage"].min(),
        "Assessment Dates": df["assessment_date"].nunique(),
        "Date Range": f"{df['assessment_date'].min()} to {df['assessment_date'].max()}",
    }

    return report


# Uncomment after data loading:
# report = generate_damage_report(df_damage)
# print("\n" + "="*50)
# print("ALEPPO BUILDING DAMAGE ASSESSMENT REPORT")
# print("="*50)
# for key, value in report.items():
#     print(f"{key}: {value}")
# print("="*50)

print("Statistical summary function defined")

Statistical summary function defined


## Damage Cause Analysis

Analyze causes of damage if metadata is available in the assessment data.

In [19]:
def analyze_damage_causes(df, cause_column="damage_cause"):
    """
    Analyze and visualize causes of damage.

    Typical UNOSAT damage causes include:
    - Airstrike/Bombing
    - Artillery
    - Shelling
    - Ground Combat
    - Secondary Destruction
    - Unspecified

    Parameters:
    -----------
    df : pd.DataFrame
        Damage assessment data
    cause_column : str
        Column name for damage cause
    """
    causes = df[cause_column].value_counts()

    fig, ax = plt.subplots(figsize=(10, 6))
    causes.plot(kind="barh", ax=ax, color="#A64D33", alpha=0.7)
    ax.set_xlabel("Number of Incidents", fontsize=12)
    ax.set_title("Building Damage by Cause", fontsize=14, fontweight="bold")
    ax.grid(True, alpha=0.3, axis="x")

    plt.tight_layout()

    return causes, fig


def analyze_cause_by_neighborhood(
    df, neighborhood_column="neighborhood", cause_column="damage_cause"
):
    """
    Create a breakdown of damage causes by neighborhood.

    Parameters:
    -----------
    df : pd.DataFrame
        Damage assessment data
    neighborhood_column : str
        Column for neighborhood names
    cause_column : str
        Column for damage causes
    """
    # Create cross-tabulation
    cause_by_neighborhood = pd.crosstab(df[neighborhood_column], df[cause_column])

    return cause_by_neighborhood


# Uncomment after data loading:
# if 'damage_cause' in df_damage.columns:
#     causes, fig = analyze_damage_causes(df_damage)
#     plt.show()
#
#     cause_breakdown = analyze_cause_by_neighborhood(df_damage)
#     print("\nDamage Causes by Top 10 Neighborhoods:")
#     print(cause_breakdown.head(10))

print("Damage cause analysis functions defined")

Damage cause analysis functions defined


## Conclusion

This notebook provides a comprehensive framework for analyzing:

1. **Temporal Progression**: How damage accumulated over time with each UNOSAT assessment
2. **Spatial Patterns**: Which neighborhoods experienced the most damage
3. **Damage Severity**: Distribution of damage levels across the city
4. **Damage Types**: Classification of damage from light to destroyed
5. **Damage Causes**: Analysis of what caused the damage (when available)

**Next Steps:**
- Load your actual UNOSAT damage assessment data
- Run the analysis pipeline sections step-by-step
- Customize visualizations for your presentation
- Export results as maps and reports